In [ ]:
import pandas as pd

# ==========================================
# 1. LOAD DATA
# ==========================================
# Path to your Training Data
train_path = "/kaggle/input/samuval/TRAIN_RELEASE_3SEP2025/train_subtask1.csv"
# Path to the Competition Test Data (Update this if your path is different)
test_path = "/kaggle/input/samevaltest/test_subtask1.csv"

# Load Train
try:
    train_df = pd.read_csv(train_path)
    print(f"✅ Loaded Train: {len(train_df)} rows")
except FileNotFoundError:
    # Fallback if custom path fails
    train_df = pd.read_csv("/kaggle/input/sem-eval-2025-task-11/train.csv")
    print("⚠️ Custom train path not found, loaded standard train.csv")

# Load Test
try:
    test_df = pd.read_csv(test_path)
    print(f"✅ Loaded Test:  {len(test_df)} rows")
except FileNotFoundError:
    print("❌ Test file not found! Please check the path.")

# ==========================================
# 2. CALCULATE OVERLAP (SEEN vs UNSEEN)
# ==========================================
train_users = set(train_df['user_id'].unique())
test_users = set(test_df['user_id'].unique())

# Logic:
# Seen = Users appearing in BOTH Train and Test
# Unseen = Users appearing ONLY in Test
seen_users = test_users.intersection(train_users)
unseen_users = test_users - train_users

# Label each row in Test
test_df['user_type'] = test_df['user_id'].apply(lambda u: 'Seen' if u in seen_users else 'Unseen')
text_counts = test_df['user_type'].value_counts()

# ==========================================
# 3. PRINT REPORT
# ==========================================
print("\n" + "="*50)
print("📊 FINAL SPLIT STATISTICS (The 'Generalization Gap')")
print("="*50)

# User Stats
n_total_users = len(test_users)
n_seen_users = len(seen_users)
n_unseen_users = len(unseen_users)

print(f"👥  Total Users in Test: {n_total_users}")
print(f"   ├─ Seen Users (Old):    {n_seen_users} ({n_seen_users/n_total_users*100:.1f}%)")
print(f"   └─ Unseen Users (New):  {n_unseen_users} ({n_unseen_users/n_total_users*100:.1f}%)")

# Text Stats
n_total_texts = len(test_df)
n_seen_texts = text_counts.get('Seen', 0)
n_unseen_texts = text_counts.get('Unseen', 0)

print(f"\n📝  Total Texts in Test: {n_total_texts}")
print(f"   ├─ Seen Texts:          {n_seen_texts} ({n_seen_texts/n_total_texts*100:.1f}%)")
print(f"   └─ Unseen Texts:        {n_unseen_texts} ({n_unseen_texts/n_total_texts*100:.1f}%)")
print("="*50)

In [ ]:
# ==========================================
# 🧹 STEP 0: DATA SANITATION (THE PURGE)
# ==========================================
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import os

print("🧹 INITIATING DATA SANITATION PROTOCOL...")

# 1. SETUP
class SanConfig:
    MODEL = "microsoft/deberta-v3-base"
    BATCH = 32
    FOLDS = 3
    POISON_THRESHOLD = 1.5 # Error > 1.5 means label is likely inverted
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    TRAIN_PATH = "/kaggle/input/samuval/TRAIN_RELEASE_3SEP2025/train_subtask1.csv"

# 2. LOAD DATA
try:
    df_full = pd.read_csv(SanConfig.TRAIN_PATH)
    print(f"   -> Loaded Full Data: {len(df_full)} rows")
except:
    print("❌ Error: Train file not found. Check path.")
    raise

# 3. HELPER CLASSES
class SimpleDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['text'].fillna("").tolist()
        self.labels = df['valence'].values
        self.tokenizer = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), padding="max_length", truncation=True, max_length=128, return_tensors="pt")
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.float)
        }

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(SanConfig.MODEL)
        self.head = nn.Linear(768, 1)
    def forward(self, input_ids, mask):
        out = self.encoder(input_ids, mask).last_hidden_state[:, 0, :]
        return self.head(out).squeeze(-1)

# 4. FIND THE POISON (OOF PREDICTIONS)
kfold = KFold(n_splits=SanConfig.FOLDS, shuffle=True, random_state=42)
tokenizer = AutoTokenizer.from_pretrained(SanConfig.MODEL)
oof_preds = np.zeros(len(df_full))

print(f"   -> Scanning for label errors ({SanConfig.FOLDS} Folds)...")

for fold, (train_idx, val_idx) in enumerate(kfold.split(df_full)):
    # Data Splits
    train_sub = df_full.iloc[train_idx].reset_index(drop=True)
    val_sub = df_full.iloc[val_idx].reset_index(drop=True)
    
    train_dl = DataLoader(SimpleDataset(train_sub, tokenizer), batch_size=16, shuffle=True)
    val_dl = DataLoader(SimpleDataset(val_sub, tokenizer), batch_size=SanConfig.BATCH, shuffle=False)
    
    # Quick Training (1 Epoch is enough to learn "Normal" patterns)
    model = SimpleModel().to(SanConfig.DEVICE)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    loss_fn = nn.MSELoss()
    
    model.train()
    for batch in tqdm(train_dl, desc=f"Fold {fold+1} Training"):
        input_ids = batch['input_ids'].to(SanConfig.DEVICE)
        mask = batch['attention_mask'].to(SanConfig.DEVICE)
        labels = batch['label'].to(SanConfig.DEVICE)
        optimizer.zero_grad()
        loss_fn(model(input_ids, mask), labels).backward()
        optimizer.step()
        
    # Predict
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in val_dl:
            input_ids = batch['input_ids'].to(SanConfig.DEVICE)
            mask = batch['attention_mask'].to(SanConfig.DEVICE)
            preds.extend(model(input_ids, mask).cpu().numpy())
    
    oof_preds[val_idx] = preds

# 5. REMOVE POISON & SAVE
df_full['oof_pred'] = oof_preds
df_full['error'] = np.abs(df_full['valence'] - df_full['oof_pred'])

# Filter
poison_mask = df_full['error'] > SanConfig.POISON_THRESHOLD
df_clean = df_full[~poison_mask].copy()

print("\n" + "="*50)
print("🧪 SANITATION REPORT")
print("="*50)
print(f"Original Size: {len(df_full)}")
print(f"Poison Removed: {poison_mask.sum()} ({(poison_mask.sum()/len(df_full))*100:.2f}%)")
print(f"Clean Size:    {len(df_clean)}")
print("-" * 50)
print("Saved to: 'train_sanitized.csv'")

df_clean.to_csv("train_sanitized.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np

# Load your files
train = pd.read_csv("/kaggle/input/samuval/TRAIN_RELEASE_3SEP2025/train_subtask1.csv")
test = pd.read_csv("/kaggle/input/samevaltest/test_subtask1.csv") # Or whatever your test file is

def get_stats(df, name):
    print(f"--- {name} ---")
    print(f"Users:    {df['user_id'].nunique()}")
    print(f"Samples:  {len(df)}")
    # Approximate length by spaces
    avg_len = df['text'].astype(str).apply(lambda x: len(x.split())).mean()
    print(f"Avg Len:  {avg_len:.1f}")
    
    if 'valence' in df:
        v_mean = df['valence'].mean()
        v_std = df['valence'].std()
        a_mean = df['arousal'].mean()
        a_std = df['arousal'].std()
        print(f"Valence:  {v_mean:.2f} ± {v_std:.2f}")
        print(f"Arousal:  {a_mean:.2f} ± {a_std:.2f}")
    else:
        print("Labels:   N/A (Test Set)")

get_stats(train, "TRAIN")
get_stats(test, "TEST")

In [ ]:
# ==========================================
# 🧹 GPU MEMORY CLEANUP
# ==========================================
import torch
import gc

# 1. Delete the Python objects holding the memory
try:
    del model, opt, crit, train_dl, val_dl
    print("Deleted model and data loaders.")
except NameError:
    print("Variables already deleted or not defined.")

# 2. Force Python Garbage Collection
gc.collect()

# 3. Clear PyTorch CUDA Cache
torch.cuda.empty_cache()

# 4. Check status
print(f"Memory Freed. Current Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

In [ ]:
# ==========================================
# 🏆 RESTORING THE CHAMPION: AROUSAL DANN
#    - Back to Stateless (No Context)
#    - DANN Architecture (Fixes Between-User)
#    - Unified Loss (Fixes Within-User)
# ==========================================
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr
import gc

# --- CONFIG ---
class Config:
    MODEL_NAME = "microsoft/deberta-v3-base"
    BATCH_SIZE = 16       # Back to standard batch size
    LR = 2e-5
    EPOCHS = 4
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    TRAIN_FILE = "train_sanitized.csv"

# --- LOSS (MSE + CCC + Adversarial) ---
class GrandLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()
        self.ce = nn.CrossEntropyLoss()
        
    def ccc(self, y_pred, y_true):
        y_true = y_true.view(-1)
        y_pred = y_pred.view(-1)
        mu_x, mu_y = y_true.mean(), y_pred.mean()
        var_x, var_y = y_true.var(), y_pred.var()
        cov = torch.mean((y_true - mu_x) * (y_pred - mu_y))
        ccc_val = (2 * cov) / (var_x + var_y + (mu_x - mu_y)**2 + 1e-8)
        return 1.0 - ccc_val

    def forward(self, p_a, y_a, p_u, y_u):
        # Arousal Loss: 50% MSE (Precision) + 50% CCC (Correlation/Trend)
        l_reg = 0.5 * self.mse(p_a.view(-1), y_a.view(-1)) + 0.5 * self.ccc(p_a, y_a)
        # Adversarial Loss: Predict User ID
        l_adv = self.ce(p_u, y_u)
        return l_reg + l_adv

# --- MODEL (Stateless DANN) ---
class ArousalDANN_Base(nn.Module):
    def __init__(self, num_users):
        super().__init__()
        self.base = AutoModel.from_pretrained(Config.MODEL_NAME)
        # Regression Head
        self.head = nn.Sequential(
            nn.Linear(768, 256), 
            nn.BatchNorm1d(256), 
            nn.ReLU(), 
            nn.Dropout(0.2), 
            nn.Linear(256, 1)
        )
        # Adversarial Head (User Classifier)
        self.adv = nn.Sequential(
            nn.Linear(768, 256), 
            nn.ReLU(), 
            nn.Linear(256, num_users)
        )
        
    def forward(self, ids, mask, alpha=1.0):
        # Gradient Reversal Layer (GRL)
        class GRL(torch.autograd.Function):
            @staticmethod
            def forward(ctx, x, a):
                ctx.a = a
                return x.view_as(x)
            @staticmethod
            def backward(ctx, g):
                return g.neg() * ctx.a, None
                
        outputs = self.base(ids, mask)
        emb = outputs.last_hidden_state[:, 0, :] # [CLS] token
        
        # Return: (Arousal Prediction, User Prediction)
        return self.head(emb), self.adv(GRL.apply(emb, alpha))

# --- DATASET ---
full_df = pd.read_csv(Config.TRAIN_FILE)
# Standard Random Split allows DANN to see all users (essential for DANN training)
train_split, val_split = train_test_split(full_df, test_size=0.1, random_state=42)

class UniDataset(Dataset):
    def __init__(self, df, tokenizer, all_users):
        self.txt = df['text'].values
        self.lbl = df['arousal'].values
        self.uid = df['user_id'].values
        self.tokenizer = tokenizer
        # Map ALL users to 0..N IDs so dimensions match
        self.umap = {u: i for i, u in enumerate(all_users)}
        
    def __len__(self): return len(self.txt)
    
    def __getitem__(self, i):
        enc = self.tokenizer(str(self.txt[i]), truncation=True, padding='max_length', max_length=128, return_tensors='pt')
        u_idx = self.umap.get(self.uid[i], 0) # Safety get
        return {
            'ids': enc['input_ids'][0], 
            'mask': enc['attention_mask'][0],
            'y': torch.tensor(self.lbl[i], dtype=torch.float),
            'u': torch.tensor(u_idx, dtype=torch.long)
        }

tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
all_users_list = full_df['user_id'].unique() # Pass this to ensure consistency
train_dl = DataLoader(UniDataset(train_split, tokenizer, all_users_list), batch_size=Config.BATCH_SIZE, shuffle=True, drop_last=True)
val_dl = DataLoader(UniDataset(val_split, tokenizer, all_users_list), batch_size=32, shuffle=False)

# --- TRAINING LOOP ---
print("🚀 RESTORING AROUSAL MODEL (Target: ~0.43 Composite)...")
model = ArousalDANN_Base(len(all_users_list)).to(Config.DEVICE)
opt = AdamW(model.parameters(), lr=Config.LR)
crit = GrandLoss()

for epoch in range(Config.EPOCHS):
    model.train()
    loss_tot = 0
    # Ramp up alpha (0 -> 1) to let regression learn first, then adversary kicks in
    p = float(epoch) / Config.EPOCHS
    alpha = 2.0 / (1.0 + np.exp(-10 * p)) - 1
    
    for b in tqdm(train_dl, desc=f"Epoch {epoch+1}"):
        ids, mask = b['ids'].to(Config.DEVICE), b['mask'].to(Config.DEVICE)
        opt.zero_grad()
        
        p_a, p_u = model(ids, mask, alpha)
        loss = crit(p_a, b['y'].to(Config.DEVICE), p_u, b['u'].to(Config.DEVICE))
        
        loss.backward()
        opt.step()
        loss_tot += loss.item()
        
    print(f"   Avg Loss: {loss_tot/len(train_dl):.4f}")

# --- EVALUATION ---
print("\n📊 CALCULATING DETAILED SCORES...")
model.eval()
preds, trues, users = [], [], []

with torch.no_grad():
    for b in val_dl:
        ids, mask = b['ids'].to(Config.DEVICE), b['mask'].to(Config.DEVICE)
        # alpha=0 disables adversarial confusion during inference
        p, _ = model(ids, mask, alpha=0)
        preds.extend(p.flatten().cpu().numpy())
        trues.extend(b['y'].flatten().numpy())
        # We need user IDs for the metric calculation
        # Re-extract them from the dataset logic for the current batch
        # (Simplified: we rely on val_split order matching dataloader order because shuffle=False)

# Re-attach predictions to the validation dataframe for grouping
df_res = val_split.copy()
df_res['prediction'] = preds

# Metric Calculation (Official SemEval Logic)
def calculate_metrics(df):
    # 1. Between-User (Personality)
    # Average preds per user, then correlate with average truth
    user_means = df.groupby('user_id')[['arousal', 'prediction']].mean()
    if len(user_means) > 1:
        r_between = pearsonr(user_means['arousal'], user_means['prediction'])[0]
    else:
        r_between = 0.0
        
    # 2. Within-User (Mood)
    # Correlate per user, then average the correlations
    r_list = []
    for user in df.user_id.unique():
        s = df[df.user_id == user]
        # Need >2 samples and some variance to calculate correlation
        if len(s) > 2 and s['arousal'].std() > 1e-9:
            r, _ = pearsonr(s['arousal'], s['prediction'])
            r_list.append(r)
            
    r_within = np.mean(r_list) if r_list else 0.0
    
    # 3. Composite (Fisher Mean)
    # Tanh of average of arctanh
    z_b = np.arctanh(np.clip(r_between, -0.99, 0.99))
    z_w = np.arctanh(np.clip(r_within, -0.99, 0.99))
    r_composite = np.tanh((z_b + z_w) / 2)
    
    return r_between, r_within, r_composite

rb, rw, rc = calculate_metrics(df_res)

print("-" * 40)
print(f"🏆 FINAL EVALUATION REPORT")
print("-" * 40)
print(f"👥 Between-User R (Personality): {rb:.4f}")
print(f"👤 Within-User R  (Mood):        {rw:.4f}")
print("-" * 40)
print(f"✅ COMPOSITE SCORE:              {rc:.4f} (Expect > 0.43)")
print("-" * 40)

# Save
torch.save(model.state_dict(), "arousal_base_unified.pth")
print("Model saved as 'arousal_base_unified.pth'")

In [ ]:
# ==========================================
# 🏆 FIXED VALENCE: BASE + MSE + ISOTONIC
#    - Metric Update: Uses Official SemEval Scorer (Fisher-Z + MAE)
# ==========================================
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error # <--- Added for Official Score

# --- CONFIG ---
class ValConfig:
    MODEL = "microsoft/deberta-v3-base"
    BATCH = 16
    LR = 2e-5
    EPOCHS = 3
    DEVICE = "cuda"
    TRAIN_FILE = "train_sanitized.csv"

# --- DATASET ---
class ValenceDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.txt = df['text'].values
        self.lbl = df['valence'].values
        self.uid = df['user_id'].values
        self.tokenizer = tokenizer
        
    def __len__(self): return len(self.txt)
    def __getitem__(self, i):
        enc = self.tokenizer(str(self.txt[i]), truncation=True, padding='max_length', max_length=128, return_tensors='pt')
        return {
            'ids': enc['input_ids'][0], 'mask': enc['attention_mask'][0],
            'y': torch.tensor(self.lbl[i], dtype=torch.float),
            'u': self.uid[i]
        }

# --- MODEL ---
class ValenceModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = AutoModel.from_pretrained(ValConfig.MODEL)
        self.head = nn.Linear(768, 1) 
        
    def forward(self, ids, mask):
        emb = self.base(ids, mask).last_hidden_state[:, 0, :]
        return self.head(emb)

# --- SETUP ---
print("🚀 TRAINING VALENCE (Official Eval Mode)...")
full_df = pd.read_csv(ValConfig.TRAIN_FILE)
train_split, val_split = train_test_split(full_df, test_size=0.1, random_state=42)
tokenizer = AutoTokenizer.from_pretrained(ValConfig.MODEL)

train_dl = DataLoader(ValenceDataset(train_split, tokenizer), batch_size=ValConfig.BATCH, shuffle=True)
val_dl = DataLoader(ValenceDataset(val_split, tokenizer), batch_size=32, shuffle=False)

model_val = ValenceModel().to(ValConfig.DEVICE)
opt = AdamW(model_val.parameters(), lr=ValConfig.LR)
crit = nn.MSELoss()

# --- TRAINING LOOP ---
for epoch in range(ValConfig.EPOCHS):
    model_val.train()
    loss_tot = 0
    for b in tqdm(train_dl, desc=f"Epoch {epoch+1}"):
        ids, mask = b['ids'].to(ValConfig.DEVICE), b['mask'].to(ValConfig.DEVICE)
        y = b['y'].to(ValConfig.DEVICE)
        
        opt.zero_grad()
        pred = model_val(ids, mask).squeeze()
        loss = crit(pred, y)
        loss.backward()
        opt.step()
        loss_tot += loss.item()
    print(f"   Valence Loss: {loss_tot/len(train_dl):.4f}")

# --- VALIDATION ---
print("\n📊 GENERATING PREDICTIONS...")
model_val.eval()
val_preds, val_true, val_uids = [], [], []

with torch.no_grad():
    for b in tqdm(val_dl):
        ids, mask = b['ids'].to(ValConfig.DEVICE), b['mask'].to(ValConfig.DEVICE)
        p = model_val(ids, mask).squeeze()
        val_preds.extend(p.cpu().numpy())
        val_true.extend(b['y'].cpu().numpy())
        val_uids.extend(b['u'])

val_preds = np.array(val_preds)
val_true = np.array(val_true)

# --- ISOTONIC FIT ---
# Note: Fitting on Val is slightly optimistic but good for estimating potential
iso_reg = IsotonicRegression(out_of_bounds='clip')
iso_reg.fit(val_preds, val_true)
iso_preds = iso_reg.transform(val_preds)

# --- OFFICIAL SCORER FUNCTION ---
def get_official_metrics(df, true_col, pred_col):
    # 1. Between-User (Pearson & MAE)
    u_means = df.groupby('user_id')[[true_col, pred_col]].mean()
    if len(u_means) > 1:
        rb = pearsonr(u_means[true_col], u_means[pred_col])[0]
        mae_b = mean_absolute_error(u_means[true_col], u_means[pred_col])
    else:
        rb, mae_b = 0.0, 0.0

    # 2. Within-User (Pearson & MAE)
    r_list, mae_list = [], []
    for u in df['user_id'].unique():
        s = df[df['user_id'] == u]
        
        # Pearson: Only if variance exists (Safe from Isotonic constant steps)
        if len(s) > 2 and s[true_col].std() > 1e-9 and s[pred_col].std() > 1e-9:
            r_list.append(pearsonr(s[true_col], s[pred_col])[0])
            
        # MAE: Always calc
        if len(s) > 0:
            mae_list.append(mean_absolute_error(s[true_col], s[pred_col]))
            
    rw = np.mean(r_list) if r_list else 0.0
    mae_w = np.mean(mae_list) if mae_list else 0.0

    # 3. Composite (Fisher-Z Transformed)
    def fisher_avg(v1, v2):
        # Clip to avoid infinity
        v1 = min(max(v1, -0.999), 0.999)
        v2 = min(max(v2, -0.999), 0.999)
        z1 = np.arctanh(v1)
        z2 = np.arctanh(v2)
        return np.tanh((z1 + z2) / 2)
    
    rc = fisher_avg(rb, rw)
    
    return rb, rw, rc, mae_b, mae_w

# --- REPORTING ---
df_res = pd.DataFrame({'user_id': val_uids, 'gold': val_true, 'raw': val_preds, 'iso': iso_preds})

# Calculate for Raw
rb_raw, rw_raw, rc_raw, maeb_raw, maew_raw = get_official_metrics(df_res, 'gold', 'raw')

# Calculate for Isotonic
rb_iso, rw_iso, rc_iso, maeb_iso, maew_iso = get_official_metrics(df_res, 'gold', 'iso')

print("\n" + "="*75)
print(f"🎭 OFFICIAL VALENCE RESULTS (Fisher-Z Composite)")
print("="*75)
print(f"{'METRIC':<20} | {'RAW MODEL':<22} | {'ISOTONIC':<22}")
print("-" * 75)
print(f"{'Between-User (r)':<20} | {rb_raw:.4f}               | {rb_iso:.4f}")
print(f"{'Within-User (r)':<20} | {rw_raw:.4f}               | {rw_iso:.4f}")
print("-" * 75)
print(f"{'⭐ COMPOSITE (r)':<20} | {rc_raw:.4f}               | {rc_iso:.4f}")
print("-" * 75)
print(f"{'Between-User (MAE)':<20} | {maeb_raw:.4f}               | {maeb_iso:.4f}")
print(f"{'Within-User (MAE)':<20} | {maew_raw:.4f}               | {maew_iso:.4f}")
print("="*75)

# Save
torch.save(model_val.state_dict(), "valence_base_final.pth")
import pickle
with open("valence_iso.pkl", "wb") as f:
    pickle.dump(iso_reg, f)

In [ ]:
# ==========================================
# 🎨 GENERATE t-SNE PLOT FOR THE PAPER
# ==========================================
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

# --- 1. MODEL DEFINITION (Copied from your Inference Cell) ---
class ArousalDANN_Base(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = AutoModel.from_pretrained("microsoft/deberta-v3-base")
        self.head = nn.Sequential(
            nn.Linear(768, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2), 
            nn.Linear(256, 1)
        )
        # Dummy adversary layer (needed only to load weights safely)
        self.adv = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Linear(256, 1))

    def forward(self, ids, mask):
        emb = self.base(ids, mask).last_hidden_state[:, 0, :]
        return self.head(emb)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. LOAD DATA & SELECT TOP 5 USERS ---
print("📂 Loading data and selecting Top 5 users...")
train_clean = pd.read_csv("train_sanitized.csv")
top_users = train_clean['user_id'].value_counts().head(5).index
plot_df = train_clean[train_clean['user_id'].isin(top_users)].copy()
print(f"Selected {len(plot_df)} texts from users: {list(top_users)}")

# --- 3. LOAD TRAINED MODEL WEIGHTS ---
print("⚡ Loading Arousal DANN Model...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
model_aro = ArousalDANN_Base().to(DEVICE)

# Load weights exactly like Cell 6 to avoid mismatch errors
state = torch.load("arousal_base_unified.pth", map_location=DEVICE)
state = {k: v for k, v in state.items() if "adv" not in k}
model_aro.load_state_dict(state, strict=False)
model_aro.eval()

# --- 4. EXTRACT LATENT EMBEDDINGS (h_i) ---
embeddings = []
user_labels = []

print("🧠 Extracting DeBERTa latent features...")
with torch.no_grad():
    for _, row in tqdm(plot_df.iterrows(), total=len(plot_df)):
        # Tokenize the text
        inputs = tokenizer(str(row['text']), truncation=True, max_length=128, 
                           padding='max_length', return_tensors='pt').to(DEVICE)
        
        # Bypass the regression head; grab the [CLS] embedding directly from the base
        latent_vector = model_aro.base(**inputs).last_hidden_state[:, 0, :] 
        
        embeddings.append(latent_vector.squeeze().cpu().numpy())
        user_labels.append(row['user_id'])

embeddings = np.array(embeddings)

# --- 5. RUN t-SNE ---
print("📉 Running t-SNE Dimensionality Reduction...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca', learning_rate='auto')
tsne_results = tsne.fit_transform(embeddings)

plot_df['tsne-2d-one'] = tsne_results[:,0]
plot_df['tsne-2d-two'] = tsne_results[:,1]

# --- 6. PLOT FORMATTING FOR ACADEMIC PAPER ---
print("🎨 Generating Plot...")
plt.figure(figsize=(8, 6))
sns.set_theme(style="whitegrid")

scatter = sns.scatterplot(
    x="tsne-2d-one", y="tsne-2d-two",
    hue="user_id",
    palette=sns.color_palette("husl", len(top_users)),
    data=plot_df,
    legend="full",
    alpha=0.7,
    s=70, # Marker size
    edgecolor=None
)

# Clean up aesthetics
plt.title("t-SNE Projection of Arousal Latent Space ($h_i$)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("t-SNE Dimension 1", fontsize=12)
plt.ylabel("t-SNE Dimension 2", fontsize=12)

# Format the legend cleanly outside the plot
plt.legend(title="User ID", bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
sns.despine(left=True, bottom=True) 

# --- 7. SAVE ---
plt.tight_layout()
plt.savefig("tsne_plot.png", dpi=300, bbox_inches='tight')
print("✅ SUCCESS: Saved as 'tsne_plot.png' at 300 DPI. Ready for Overleaf!")
plt.show()

In [ ]:
# ==========================================
# 📦 GENERATE SUBMISSION (Strict Format)
#    - Filename: pred_subtask1.csv
#    - Columns: user_id, text_id, pred_valence, pred_arousal
# ==========================================
import torch
import pandas as pd
import numpy as np
import zipfile
import pickle
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn

# --- CONFIG ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
TEST_PATH = "/kaggle/input/samevaltest/test_subtask1.csv"
OUTPUT_CSV = "pred_subtask1.csv"  # <--- REQUIRED FILENAME
OUTPUT_ZIP = "submission.zip"

# --- MODEL DEFINITIONS ---
class ArousalDANN_Base(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = AutoModel.from_pretrained("microsoft/deberta-v3-base")
        self.head = nn.Sequential(
            nn.Linear(768, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2), 
            nn.Linear(256, 1)
        )
        # Dummy adversary layer (needed only to load weights safely)
        self.adv = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Linear(256, 1))

    def forward(self, ids, mask):
        emb = self.base(ids, mask).last_hidden_state[:, 0, :]
        return self.head(emb)

class ValenceModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = AutoModel.from_pretrained("microsoft/deberta-v3-base")
        self.head = nn.Linear(768, 1) 
    def forward(self, ids, mask):
        emb = self.base(ids, mask).last_hidden_state[:, 0, :]
        return self.head(emb)

# --- DATASET ---
class TestDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.txt = df['text'].astype(str).values
        self.tokenizer = tokenizer
    def __len__(self): return len(self.txt)
    def __getitem__(self, i):
        enc = self.tokenizer(self.txt[i], truncation=True, max_length=128, padding='max_length', return_tensors='pt')
        return {'ids': enc['input_ids'][0], 'mask': enc['attention_mask'][0]}

# ==========================
# 🚀 INFERENCE
# ==========================
print("🚀 STARTING FINAL INFERENCE...")

# 1. PREPARE DATA
test_df = pd.read_csv(TEST_PATH)
def sanitize_text(text): return str(text).replace("\n", " ").strip()
test_df['text'] = test_df['text'].apply(sanitize_text)

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
test_dl = DataLoader(TestDataset(test_df, tokenizer), batch_size=BATCH_SIZE, shuffle=False)

# 2. PREDICT AROUSAL
print("⚡ Loading Arousal Model...")
aro_preds = np.zeros(len(test_df))
try:
    model_aro = ArousalDANN_Base().to(DEVICE)
    # Load weights, ignoring the adversary size mismatch if strictly necessary
    state = torch.load("arousal_base_unified.pth", map_location=DEVICE)
    # Filter out adversary keys to prevent shape mismatch errors
    state = {k: v for k, v in state.items() if "adv" not in k}
    model_aro.load_state_dict(state, strict=False)
    model_aro.eval()
    
    preds = []
    with torch.no_grad():
        for b in tqdm(test_dl, desc="Predicting Arousal"):
            ids, mask = b['ids'].to(DEVICE), b['mask'].to(DEVICE)
            p = model_aro(ids, mask)
            preds.extend(p.flatten().cpu().numpy())
    aro_preds = np.array(preds)
except Exception as e:
    print(f"❌ ERROR: Could not load Arousal model ({e})")

# 3. PREDICT VALENCE
print("🎭 Loading Valence Model...")
val_preds = np.zeros(len(test_df))
try:
    model_val = ValenceModel().to(DEVICE)
    model_val.load_state_dict(torch.load("valence_base_final.pth", map_location=DEVICE))
    model_val.eval()
    
    with open("valence_iso.pkl", "rb") as f:
        iso_reg = pickle.load(f)
        
    raw_preds = []
    with torch.no_grad():
        for b in tqdm(test_dl, desc="Predicting Valence"):
            ids, mask = b['ids'].to(DEVICE), b['mask'].to(DEVICE)
            p = model_val(ids, mask).squeeze()
            raw_preds.extend(p.cpu().numpy())
    
    # Apply Isotonic Calibration
    val_preds = iso_reg.transform(np.array(raw_preds))
except Exception as e:
    print(f"❌ ERROR: Could not load Valence model ({e})")

# ==========================
# 💾 FORMATTING & ZIPPING
# ==========================
print("\n📦 PACKAGING SUBMISSION...")

sub = pd.DataFrame()
# 1. Column Order MUST be exact: user_id, text_id, pred_valence, pred_arousal
sub['user_id'] = test_df['user_id']
sub['text_id'] = test_df['text_id']
sub['pred_valence'] = np.clip(val_preds, -2, 2)
sub['pred_arousal'] = np.clip(aro_preds, 0, 2)

# 2. Check for NaN
if sub.isnull().values.any():
    print("⚠️ WARNING: NaNs detected! Filling with means...")
    sub['pred_valence'] = sub['pred_valence'].fillna(0.5)
    sub['pred_arousal'] = sub['pred_arousal'].fillna(0.7)

# 3. Save CSV
sub.to_csv(OUTPUT_CSV, index=False)
print(f"   Saved '{OUTPUT_CSV}' with shape {sub.shape}")

# 4. Create ZIP
with zipfile.ZipFile(OUTPUT_ZIP, 'w') as z:
    z.write(OUTPUT_CSV)

print(f"✅ DONE. '{OUTPUT_ZIP}' is ready to upload!")
print("\n👀 HEADER CHECK (First 3 rows):")
print(sub.head(3))

In [ ]:
# ============================================
# 🕵️ FORENSIC ANALYSIS: VALENCE ISOTONIC CHECK (FIXED)
# ============================================
import torch
import torch.nn as nn
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# --- 1. CONFIG & CLASSES ---
class ValConfig:
    MODEL = "microsoft/deberta-v3-base"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    TRAIN_FILE = "train_sanitized.csv"

class ValenceModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = AutoModel.from_pretrained(ValConfig.MODEL)
        self.head = nn.Linear(768, 1)
    def forward(self, ids, mask):
        emb = self.base(ids, mask).last_hidden_state[:, 0, :]
        return self.head(emb)

class ValenceDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.txt = df['text'].values
        # Handle labels safely
        self.lbl = df['valence'].values if 'valence' in df else np.zeros(len(df))
        self.tokenizer = tokenizer  # <--- FIXED: Now actually storing the tokenizer
        
    def __len__(self): return len(self.txt)
    
    def __getitem__(self, i):
        enc = self.tokenizer(str(self.txt[i]), truncation=True, padding='max_length', max_length=128, return_tensors='pt')
        return {
            'ids': enc['input_ids'][0],
            'mask': enc['attention_mask'][0],
            'y': torch.tensor(self.lbl[i], dtype=torch.float)
        }

# --- 2. REBUILD VALIDATION SET ---
print("♻️ Reconstructing Validation Set (State=42)...")
full_df = pd.read_csv(ValConfig.TRAIN_FILE)
_, val_split = train_test_split(full_df, test_size=0.1, random_state=42)

tokenizer = AutoTokenizer.from_pretrained(ValConfig.MODEL)
# Now this works because the class is fixed
val_ds = ValenceDataset(val_split, tokenizer)
val_dl = DataLoader(val_ds, batch_size=32, shuffle=False)

# --- 3. LOAD MODELS ---
print("Weight Loading...")
model = ValenceModel().to(ValConfig.DEVICE)
model.load_state_dict(torch.load("valence_base_final.pth", map_location=ValConfig.DEVICE))
model.eval()

with open("valence_iso.pkl", "rb") as f:
    iso_reg = pickle.load(f)
print("✅ Models Loaded.")

# --- 4. GENERATE PREDICTIONS & GET TRUTH ---
print("\nGenerating Raw Predictions & Grabbing Truth...")
raw_preds = []
val_true = []

with torch.no_grad():
    for b in tqdm(val_dl):
        ids, mask = b['ids'].to(ValConfig.DEVICE), b['mask'].to(ValConfig.DEVICE)
        p = model(ids, mask).squeeze()
        raw_preds.extend(p.cpu().numpy())
        val_true.extend(b['y'].cpu().numpy())

raw_preds = np.array(raw_preds)
val_true = np.array(val_true)

# --- 5. APPLY ISOTONIC ---
print("Applying Isotonic Calibration...")
iso_preds = iso_reg.transform(raw_preds)

# --- 6. PLOT THE EVIDENCE ---
plt.figure(figsize=(14, 6))

# Plot A: The Histograms (Distribution Check)
plt.subplot(1, 2, 1)
# Ground Truth
plt.hist(val_true, bins=100, color='green', alpha=0.3, label='Ground Truth', density=True)
# Raw Model
plt.hist(raw_preds, bins=100, color='blue', alpha=0.3, label='Raw Model', density=True)
# Isotonic
plt.hist(iso_preds, bins=100, color='red', alpha=0.5, label='Isotonic', density=True)

plt.title("Before vs. After: Distribution Check")
plt.xlabel("Valence Score")
plt.ylabel("Density")
plt.legend()
plt.grid(True, alpha=0.3)

# Plot B: The Staircase (Quantization Check)
plt.subplot(1, 2, 2)
sort_idxs = np.argsort(raw_preds)
plt.plot(raw_preds[sort_idxs], iso_preds[sort_idxs], color='red', linewidth=2, label="Isotonic Function")
plt.plot([min(raw_preds), max(raw_preds)], [min(raw_preds), max(raw_preds)], 'k--', alpha=0.3, label="Identity (No Change)")
plt.title("The Transformation Function (Staircase Check)")
plt.xlabel("Raw Input")
plt.ylabel("Calibrated Output")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()

# --- 7. SAVE THE FIGURE ---
save_path = "valence_forensic_analysis.png"
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"\n✅ Analysis plot saved to: {save_path}")

plt.show()

# --- 8. VARIANCE STATS ---
print("\n📊 STATISTICAL EVIDENCE:")
print(f"Original Variance:   {np.var(raw_preds):.6f}")
print(f"Calibrated Variance: {np.var(iso_preds):.6f}")
print(f"Ground Truth Variance: {np.var(val_true):.6f}")
drop = (1 - np.var(iso_preds)/(np.var(raw_preds) + 1e-9)) * 100
print(f"Variance Loss:       {drop:.2f}%")

In [ ]:
# ==========================================
# 📊 ENSEMBLE VALIDATION RUN (Seeds 42, 43, 44)
#    - Validates on Fixed Val Set (No Leakage)
#    - Uses Nested Split for Valence Calibration
# ==========================================
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression
from scipy.stats import pearsonr
import gc
import random

# --- CONFIG ---
SEEDS = [42, 43, 44]
DEVICE = "cuda"
BATCH_SIZE = 16
EPOCHS_ARO = 4
EPOCHS_VAL = 3
LR = 2e-5

# --- SETUP FIXED VALIDATION SET ---
full_df = pd.read_csv("train_sanitized.csv")
# FIXED VALIDATION SET (Seed 42) - The "Truth" we measure against
train_df_main, val_df_fixed = train_test_split(full_df, test_size=0.1, random_state=42)

print(f"📊 DATA SPLIT:")
print(f"   Train Pool: {len(train_df_main)} samples")
print(f"   Fixed Val:  {len(val_df_fixed)} samples (Evaluating on this)")

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

# --- DATASETS ---
class UniDataset(Dataset):
    def __init__(self, df, tokenizer, mode='aro'):
        self.txt = df['text'].values
        self.lbl = df['arousal'].values if mode=='aro' else df['valence'].values
        self.uid = df['user_id'].values
        self.tokenizer = tokenizer
        self.umap = {u: i for i, u in enumerate(full_df['user_id'].unique())}
        self.mode = mode
    def __len__(self): return len(self.txt)
    def __getitem__(self, i):
        enc = self.tokenizer(str(self.txt[i]), truncation=True, max_length=128, padding='max_length', return_tensors='pt')
        u_idx = self.umap.get(self.uid[i], 0)
        return {
            'ids': enc['input_ids'][0], 'mask': enc['attention_mask'][0],
            'y': torch.tensor(self.lbl[i], dtype=torch.float),
            'u': torch.tensor(u_idx, dtype=torch.long)
        }

# --- MODELS ---
class ArousalDANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = AutoModel.from_pretrained("microsoft/deberta-v3-base")
        self.head = nn.Linear(768, 1)
        self.adv = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Linear(256, len(full_df.user_id.unique())))
    def forward(self, ids, mask, alpha=0):
        class GRL(torch.autograd.Function):
            @staticmethod
            def forward(ctx, x, a): ctx.a=a; return x.view_as(x)
            @staticmethod
            def backward(ctx, g): return g.neg()*ctx.a, None
        emb = self.base(ids, mask).last_hidden_state[:,0,:]
        return self.head(emb), self.adv(GRL.apply(emb, alpha))

class ValenceModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = AutoModel.from_pretrained("microsoft/deberta-v3-base")
        self.head = nn.Linear(768, 1)
    def forward(self, ids, mask):
        return self.head(self.base(ids, mask).last_hidden_state[:,0,:])

# --- LOSS ---
class ArousalLoss(nn.Module):
    def forward(self, pa, ya, pu, yu):
        mse = nn.MSELoss()(pa.view(-1), ya.view(-1))
        # CCC
        mu_x, mu_y = ya.mean(), pa.mean()
        var_x, var_y = ya.var(), pa.var()
        cov = torch.mean((ya-mu_x)*(pa-mu_y))
        ccc = 1.0 - (2*cov)/(var_x+var_y+(mu_x-mu_y)**2+1e-8)
        return 0.5*mse + 0.5*ccc + nn.CrossEntropyLoss()(pu, yu)

# --- METRIC CALCULATOR ---
def calc_metrics(df, true_col, pred_col):
    # Between
    u = df.groupby('user_id')[[true_col, pred_col]].mean()
    rb = pearsonr(u[true_col], u[pred_col])[0] if len(u)>2 else 0
    # Within
    ws = []
    for user in df.user_id.unique():
        s = df[df.user_id==user]
        if len(s)>2 and s[true_col].std()>1e-9:
            ws.append(pearsonr(s[true_col], s[pred_col])[0])
    rw = np.mean(ws) if ws else 0
    # Composite
    rc = np.tanh((np.arctanh(min(max(rb,-0.99),0.99)) + np.arctanh(min(max(rw,-0.99),0.99)))/2)
    return rb, rw, rc

# ==========================
# 🚀 MAIN LOOP
# ==========================
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# Store predictions for averaging
all_aro_preds = []
all_val_preds = []

print(f"🌟 VALIDATING 3-SEED ENSEMBLE {SEEDS}...")

for seed in SEEDS:
    print(f"\n🌱 SEED {seed} STARTED...")
    seed_everything(seed)
    
    # ---------------------------
    # 1. AROUSAL (Train on Main, Predict on Fixed Val)
    # ---------------------------
    train_dl = DataLoader(UniDataset(train_df_main, tokenizer, 'aro'), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_dl = DataLoader(UniDataset(val_df_fixed, tokenizer, 'aro'), batch_size=32, shuffle=False)
    
    model = ArousalDANN().to(DEVICE)
    opt = AdamW(model.parameters(), lr=LR)
    crit = ArousalLoss()
    
    for e in range(EPOCHS_ARO):
        model.train()
        alpha = 2.0/(1.0+np.exp(-10*(e/EPOCHS_ARO)))-1
        for b in tqdm(train_dl, desc=f"Arousal S{seed}-E{e+1}", leave=False):
            opt.zero_grad()
            pa, pu = model(b['ids'].to(DEVICE), b['mask'].to(DEVICE), alpha)
            loss = crit(pa, b['y'].to(DEVICE), pu, b['u'].to(DEVICE))
            loss.backward()
            opt.step()
            
    # Predict
    model.eval()
    preds = []
    with torch.no_grad():
        for b in val_dl:
            p, _ = model(b['ids'].to(DEVICE), b['mask'].to(DEVICE), alpha=0)
            preds.extend(p.flatten().cpu().numpy())
    all_aro_preds.append(preds)
    
    del model, opt, crit
    gc.collect(); torch.cuda.empty_cache()
    
    # ---------------------------
    # 2. VALENCE (Nested Split: Train -> Calib -> Val)
    # ---------------------------
    # Split Main Train into (Train Inner / Calibration)
    train_inner, calib_inner = train_test_split(train_df_main, test_size=0.1, random_state=seed) # Seed ensures variety
    
    train_dl = DataLoader(UniDataset(train_inner, tokenizer, 'val'), batch_size=BATCH_SIZE, shuffle=True)
    calib_dl = DataLoader(UniDataset(calib_inner, tokenizer, 'val'), batch_size=32, shuffle=False)
    val_dl = DataLoader(UniDataset(val_df_fixed, tokenizer, 'val'), batch_size=32, shuffle=False)
    
    model = ValenceModel().to(DEVICE)
    opt = AdamW(model.parameters(), lr=LR)
    crit = nn.MSELoss()
    
    for e in range(EPOCHS_VAL):
        model.train()
        for b in tqdm(train_dl, desc=f"Valence S{seed}-E{e+1}", leave=False):
            opt.zero_grad()
            p = model(b['ids'].to(DEVICE), b['mask'].to(DEVICE)).squeeze()
            loss = crit(p, b['y'].to(DEVICE))
            loss.backward()
            opt.step()
            
    # Fit Isotonic on Calibration Set
    model.eval()
    c_preds, c_true = [], []
    with torch.no_grad():
        for b in calib_dl:
            p = model(b['ids'].to(DEVICE), b['mask'].to(DEVICE)).squeeze()
            c_preds.extend(p.cpu().numpy())
            c_true.extend(b['y'].cpu().numpy())
    
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(c_preds, c_true)
    
    # Predict on Fixed Val
    v_preds = []
    with torch.no_grad():
        for b in val_dl:
            p = model(b['ids'].to(DEVICE), b['mask'].to(DEVICE)).squeeze()
            v_preds.extend(p.cpu().numpy())
            
    # Calibrate and Store
    final_v = iso.transform(v_preds)
    all_val_preds.append(final_v)
    
    del model, opt, crit, iso
    gc.collect(); torch.cuda.empty_cache()

# ==========================
# 📊 FINAL ENSEMBLE METRICS
# ==========================
print("\n" + "="*60)
print("📊 FINAL ENSEMBLE RESULTS (Averaged over 3 Seeds)")
print("="*60)

# Average the predictions
avg_aro = np.mean(all_aro_preds, axis=0)
avg_val = np.mean(all_val_preds, axis=0)

# Create Result DF
res_df = val_df_fixed.copy()
res_df['pred_arousal'] = avg_aro
res_df['pred_valence'] = avg_val

# Calculate Metrics
rb_a, rw_a, rc_a = calc_metrics(res_df, 'arousal', 'pred_arousal')
rb_v, rw_v, rc_v = calc_metrics(res_df, 'valence', 'pred_valence')

print(f"{'METRIC':<20} | {'AROUSAL':<15} | {'VALENCE':<15}")
print("-" * 60)
print(f"{'Between-User':<20} | {rb_a:.4f}          | {rb_v:.4f}")
print(f"{'Within-User':<20} | {rw_a:.4f}          | {rw_v:.4f}")
print("-" * 60)
print(f"{'⭐ COMPOSITE':<20} | {rc_a:.4f}          | {rc_v:.4f}")
print("="*60)